# Phase 8.5: exact reconstruction of the type $(1,1,2)$ record

The first Phase-8 control reached $\ell^2=1.068957\ldots$ but had only one exactly active logical class. A broader search and constrained max--min refinement moved to a point with all three class distances near $1.1547$. This notebook recognizes the limiting metric exactly, certifies its relative systole, and determines its CM status.

In [1]:
from dataclasses import fields
from fractions import Fraction
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from gkp_systole import (
    G3_TYPE_112_GAUSSIAN_FORM, MetricConvention, SystoleLedgerEntry,
    TYPE_112_CM_BLOCK, TYPE_112_EXACT_MODEL, TYPE_112_METRIC_CORE,
    compute_relative_systole, reconstruct_type112_metric_core,
    ternary_hermitian_moduli_family, write_systole_ledger,
)

## Refined numerical point

These coordinates are the reproducible output of the broader Phase-8 search followed by a constrained optimizer that maximized a common lower bound for the three logical-class distances.

In [2]:
coordinates = (
    0.6346061342618394, -0.8952583513598373, 1.1014161936589857,
    0.5742850646324845, 0.32556169437561483, -0.6187908894127108,
    0.7420321333966146, 0.6616025973285278, -0.6608779738052072,
    0.17589059128661858, 0.3663206922035179, -0.24501715822885206,
)
family = ternary_hermitian_moduli_family(G3_TYPE_112_GAUSSIAN_FORM)
numerical = family.evaluate(coordinates)
numerical_result = compute_relative_systole(
    family.alternating, numerical.metric,
    metric_convention=MetricConvention.POLARIZATION_SCALED,
)
distances = tuple(float(result.squared_distance) for result in numerical_result.class_results)
print('numerical ell^2 =', numerical.squared_systole)
print('three class distances =', distances)
print('spread =', max(distances) - min(distances))
assert min(distances) > 1.15469
assert max(distances) - min(distances) < 4e-9

numerical ell^2 = 1.154697407177327
three class distances = (1.154697407177327, 1.1546974074684804, 1.1546974104120262)
spread = 3.234699175180822e-09


## Recognize the half-integral metric core

Multiplying the numerical metric by $\sqrt3$ puts every entry close to a half-integer. Recognition is only a proposal; the subsequent exact PPAV and exhaustive closest-vector checks are the certificate.

In [3]:
reconstructed = reconstruct_type112_metric_core(numerical.metric)
expected = tuple(tuple(Fraction(value) for value in row) for row in TYPE_112_METRIC_CORE)
assert reconstructed == expected
maximum_error = max(
    abs(float(reconstructed[row][column]) / 3**0.5 - numerical.metric[row][column])
    for row in range(6) for column in range(6)
)
print('recognized G_core:')
for row in reconstructed:
    print(row)
print('maximum entrywise reconstruction error =', maximum_error)

recognized G_core:
(Fraction(12, 1), Fraction(-12, 1), Fraction(7, 1), Fraction(4, 1), Fraction(6, 1), Fraction(-11, 1))
(Fraction(-12, 1), Fraction(20, 1), Fraction(-5, 1), Fraction(-4, 1), Fraction(-10, 1), Fraction(15, 1))
(Fraction(7, 1), Fraction(-5, 1), Fraction(9, 1), Fraction(1, 2), Fraction(0, 1), Fraction(-4, 1))
(Fraction(4, 1), Fraction(-4, 1), Fraction(1, 2), Fraction(3, 1), Fraction(4, 1), Fraction(-11, 2))
(Fraction(6, 1), Fraction(-10, 1), Fraction(0, 1), Fraction(4, 1), Fraction(8, 1), Fraction(-10, 1))
(Fraction(-11, 1), Fraction(15, 1), Fraction(-4, 1), Fraction(-11, 2), Fraction(-10, 1), Fraction(15, 1))
maximum entrywise reconstruction error = 8.713500559665022e-05


## Exact PPAV and relative-systole certificate

The exact model is $G=G_{\rm core}/\sqrt3$ and $J=S/\sqrt3$. The centralized validator checks positivity, $J^2=-I$, $A$-compatibility, determinant normalization, and polarization type. The exact CVP solver then exhausts all three nonzero kernel classes.

In [4]:
model = TYPE_112_EXACT_MODEL
certificate = model.validation_certificate()
result = model.core_relative_systole()
print('polarization type =', model.polarization.type)
print('core ell^2 =', result.squared_systole)
print('physical ell^2 =', model.exact_squared_systole, '=', model.squared_systole)
print('class/lift multiplicity =', (result.class_multiplicity, result.lift_multiplicity))
print('validation certified =', certificate.certified, 'CVPs certified =', result.certified)
assert model.polarization.type == (1,1,2)
assert result.squared_systole == 2
assert result.class_multiplicity == 3
assert result.lift_multiplicity == 36
assert certificate.certified and result.certified

polarization type = (1, 1, 2)
core ell^2 = 2
physical ell^2 = 2/sqrt(3) = 1.1547005383792517
class/lift multiplicity = (3, 36)
validation certified = True CVPs certified = True


## CM isogeny certificate

The rational endomorphism $S$ satisfies $S^2=-3I$. An integral change of lattice of determinant 72 conjugates it to three copies of $\left(\begin{smallmatrix}0&-3\\1&0\end{smallmatrix}\right)$. Hence the threefold is isogenous to $E_{i\sqrt3}^3$ and is CM.

In [5]:
cm = model.cm_certificate()
print('field =', cm.field)
print('order discriminant =', cm.order_discriminant)
print('elliptic period =', cm.elliptic_period)
print('commutant dimension =', cm.commutant_dimension)
print('isogeny degree =', cm.rational_isogeny_degree)
print('block matrix:')
for row in cm.block_matrix:
    print(row)
assert cm.is_cm
assert cm.block_matrix == TYPE_112_CM_BLOCK

field = Q(sqrt(-3))
order discriminant = -12
elliptic period = i*sqrt(3)
commutant dimension = 18
isogeny degree = 72
block matrix:
(0, -3, 0, 0, 0, 0)
(1, 0, 0, 0, 0, 0)
(0, 0, 0, -3, 0, 0)
(0, 0, 1, 0, 0, 0)
(0, 0, 0, 0, 0, -3)
(0, 0, 0, 0, 1, 0)


## Update the result ledger

The Phase-8 numerical control remains in the ledger as provenance. This exact reconstruction is appended as the stronger current type-$(1,1,2)$ record. Global optimality remains open.

In [6]:
ledger_path = ROOT / 'data/phase8_result_ledger.json'
field_names = {field.name for field in fields(SystoleLedgerEntry)}
existing = [
    SystoleLedgerEntry(**{key: value for key, value in record.items() if key in field_names})
    for record in json.loads(ledger_path.read_text())
    if record['candidate_id'] != 'g3_type112_exact_reconstruction'
]
existing.append(SystoleLedgerEntry(
    candidate_id='g3_type112_exact_reconstruction', phase=9, dimension_g=3,
    polarization_type=str((1,1,2)), family='exact reconstructed compatible metric',
    cm_data='Q(sqrt(-3)); order discriminant -12; isogenous to E_(i*sqrt(3))^3',
    ell_squared_decimal=f'{model.squared_systole:.16g}',
    ell_squared_exact='2/sqrt(3)', class_multiplicity=3, lift_multiplicity=36,
    metric_convention='polarization_scaled', arithmetic_status='exact certified',
    search_status='exact reconstruction of optimized Phase-8 control',
    search_scope='broader 12D search plus constrained max-min refinement',
    notes='current project record for type (1,1,2); global optimality open',
))
write_systole_ledger(
    existing, json_path=ROOT/'data/phase8_result_ledger.json',
    csv_path=ROOT/'data/phase8_result_ledger.csv',
)
print('ledger entries =', len(existing))

ledger entries = 5


## Conclusion

The apparent generic improvement is not a non-CM counterexample. It converges to an exactly certified CM threefold with $\ell^2=2/\sqrt3$, improving the bounded Gaussian-CM record $1$. This is the same numerical distance value as the type $(1,1,3)$ Eisenstein record, but the two values belong to different polarization types and must not be interpreted as a global cross-type ranking.